# Train `hardware_failure_detection`

Trains the port-group ensemble for hardware failure detection. Uses LightGBM + XGBoost; both pick up the GPU automatically via `_gpu_utils`.

Drive layout (created by `00_setup.ipynb`):
```
/content/drive/MyDrive/Predictor/datasets/hardware_failure_detection/   # CSVs go here
/content/drive/MyDrive/Predictor/models/hardware_failure_detection/     # trained artifacts
```


## 1. Mount Drive & sync repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import pathlib, sys
REPO_URL = 'https://github.com/adx-coder/NUTRIRE.git'
REPO_DIR = pathlib.Path('/content/Predictor')
if not REPO_DIR.exists():
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull --ff-only
sys.path.insert(0, str(REPO_DIR / 'predictor' / 'backend'))


## 2. Install requirements (CPU defaults + CUDA torch)

In [ ]:
!pip install -q -r /content/Predictor/predictor/backend/requirements.txt
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121


## 3. Configure dataset / model paths

In [ ]:
import pathlib
PLUGIN_ID = 'hardware_failure_detection'
DATA_DIR  = pathlib.Path('/content/drive/MyDrive/Predictor/datasets') / PLUGIN_ID
MODEL_DIR = pathlib.Path('/content/drive/MyDrive/Predictor/models')   / PLUGIN_ID
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Data:', DATA_DIR)
print('Model:', MODEL_DIR)
assert DATA_DIR.exists(), f'Upload CSVs to {DATA_DIR} before training.'


## 4. Detect GPU (sanity check)

In [ ]:
from app.ml_plugins._gpu_utils import detect_gpu
print(detect_gpu())


## 5. Load training CSVs

In [ ]:
import pandas as pd
data = {}
data['telemetry'] = pd.read_csv(DATA_DIR / 'telemetry.csv')
print('telemetry:', data['telemetry'].shape)
data['alarms'] = pd.read_csv(DATA_DIR / 'alarms.csv')
print('alarms:', data['alarms'].shape)
data['topology'] = pd.read_csv(DATA_DIR / 'topology.csv')
print('topology:', data['topology'].shape)


## 6. Train the plugin

In [ ]:
from app.ml_plugins.registry import get_plugin
plugin = get_plugin(PLUGIN_ID)
assert plugin is not None, f'Plugin {PLUGIN_ID} not registered.'
metrics = plugin.train(data, MODEL_DIR)


## 7. Inspect metrics

In [ ]:
import json
print(json.dumps({k: v for k, v in metrics.items() if k != 'feature_names'}, indent=2, default=str))


## 8. Verify artifacts on Drive

In [ ]:
for p in sorted(MODEL_DIR.iterdir()):
    print(p.name, p.stat().st_size, 'bytes')
